In [ ]:
import csv
import re
import time
import random
from datetime import datetime
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import undetected_chromedriver as uc


# ── SEARCH PARAMETERS (change these to affect all cities) ─────────────────────
CHECKIN         = "2026-06-01"
CHECKOUT        = "2026-06-02"
NUM_ADULTS      = 2
NUM_CHILDREN    = 0
NUM_ROOMS       = 1

# Calculate nights automatically
from datetime import date
_cin  = date.fromisoformat(CHECKIN)
_cout = date.fromisoformat(CHECKOUT)
NUM_NIGHTS = (_cout - _cin).days


def setup_driver():
    options = uc.ChromeOptions()
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--disable-blink-features=AutomationControlled')

    driver = uc.Chrome(options=options, version_main=147)
    driver.execute_cdp_cmd('Page.addScriptToEvaluateOnNewDocument', {
        'source': '''
            Object.defineProperty(navigator, 'webdriver', {get: () => undefined});
            Object.defineProperty(navigator, 'plugins', {get: () => [1, 2, 3, 4, 5]});
            Object.defineProperty(navigator, 'languages', {get: () => ['en-US', 'en']});
        '''
    })
    return driver


def extract_text_safe(element, selector, attribute=None):
    try:
        if attribute:
            return element.find_element(By.CSS_SELECTOR, selector).get_attribute(attribute)
        return element.find_element(By.CSS_SELECTOR, selector).text.strip()
    except:
        return ""


def scroll_to_element(driver, element):
    """Scroll element into view so lazy-loaded content appears."""
    try:
        driver.execute_script("arguments[0].scrollIntoView({block: 'center'});", element)
        time.sleep(0.5)
    except:
        pass


def extract_amenities(driver, hotel):
    """Scroll to hotel card first, then extract amenities."""
    amenities = []
    try:
        scroll_to_element(driver, hotel)
        elements = hotel.find_elements(By.CSS_SELECTOR, "span.beb5ef4fb4")
        for elem in elements:
            text = elem.text.strip()
            if text:
                amenities.append(text)
    except:
        pass
    return ", ".join(amenities)


def get_property_type(hotel, hotel_name):
    """Read property type from card text, then guess from name."""
    property_types = [
        "Resort", "Hostel", "Apartment", "Villa",
        "Guest house", "Motel", "Inn", "Hotel"
    ]
    try:
        card_text = hotel.text
        for ptype in property_types:
            if ptype.lower() in card_text.lower():
                return ptype
    except:
        pass

    # Fallback: guess from hotel name
    name_lower = hotel_name.lower()
    if "hostel"                                        in name_lower: return "Hostel"
    if any(w in name_lower for w in ["apartment", "suite", "residence", "studio"]): return "Apartment"
    if "resort"                                        in name_lower: return "Resort"
    if "villa"                                         in name_lower: return "Villa"
    if "inn"                                           in name_lower: return "Inn"
    if "hotel"                                         in name_lower: return "Hotel"

    return "Hotel"  # default fallback


def build_url(city_ss):
    """Build Booking.com search URL from city name and global parameters."""
    base = "https://www.booking.com/searchresults.html"
    return (
        f"{base}?ss={city_ss}&lang=en-us"
        f"&checkin={CHECKIN}&checkout={CHECKOUT}"
        f"&group_adults={NUM_ADULTS}&no_rooms={NUM_ROOMS}"
        f"&group_children={NUM_CHILDREN}&dest_type=city"
    )


def scrape_hotels(city_name, url, max_pages=10):
    print("Starting driver...")
    driver = setup_driver()
    hotels_data = []

    try:
        print(f"Loading {url}...")
        driver.get(url)
        time.sleep(random.randint(5, 10))

        WebDriverWait(driver, 60).until(
            EC.presence_of_element_located(
                (By.CSS_SELECTOR, "[data-testid='property-card']"))
        )
        print("Page loaded, searching for hotels...")

        page_num = 1

        while page_num <= max_pages:
            print(f"\n--- Page {page_num} ---")

            # Scroll page to trigger lazy loading
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight/2);")
            time.sleep(1)
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(1)
            driver.execute_script("window.scrollTo(0, 0);")
            time.sleep(0.5)

            hotels = driver.find_elements(
                By.CSS_SELECTOR, "[data-testid='property-card']")
            print(f"Found {len(hotels)} hotels on page {page_num}")

            for idx, hotel in enumerate(hotels):
                try:
                    hotel_info = {
                        "city":                 city_name,
                        "hotel_name":           "",
                        "price":                "",
                        "currency":             "EGP",
                        "num_adults":           NUM_ADULTS,
                        "num_children":         NUM_CHILDREN,
                        "num_nights":           NUM_NIGHTS,
                        "rating":               "",
                        "reviews_count":        "",
                        "location":             "",
                        "distance_from_center": "",
                        "star_rating":          "",
                        "property_type":        "",
                        "room_type":            "",
                        "free_cancellation":    "",
                        "meal_plan":            "",
                        "amenities":            "",
                        "hotel_url":            "",
                        "scraped_date":         datetime.now().strftime("%Y-%m-%d")
                    }

                    # ── HOTEL NAME ────────────────────────────────────────
                    hotel_info["hotel_name"] = extract_text_safe(
                        hotel, "[data-testid='title']")

                    # ── PRICE ─────────────────────────────────────────────
                    price_elem = extract_text_safe(
                        hotel, "[data-testid='price-and-discounted-price']")
                    if price_elem:
                        price_clean = re.sub(r'[^\d]', '', price_elem)
                        hotel_info["price"] = price_clean if price_clean else ""

                    # ── RATING ────────────────────────────────────────────
                    rating_text = extract_text_safe(
                        hotel, "[data-testid='review-score']")
                    if rating_text:
                        rating_match = re.search(r'\b(\d+\.?\d*)\b', rating_text)
                        if rating_match:
                            hotel_info["rating"] = rating_match.group(1)

                    # ── REVIEWS COUNT ─────────────────────────────────────
                    if rating_text:
                        numbers = re.findall(r'[\d,]+', rating_text)
                        if len(numbers) >= 2:
                            hotel_info["reviews_count"] = numbers[-1].replace(",", "")

                    # ── LOCATION ──────────────────────────────────────────
                    location = extract_text_safe(
                        hotel, "[data-testid='address-link']")
                    if not location:
                        location = city_name
                    hotel_info["location"] = location

                    # ── DISTANCE ──────────────────────────────────────────
                    distance = extract_text_safe(
                        hotel, "[data-testid='distance']")
                    if distance:
                        hotel_info["distance_from_center"] = distance

                    # ── STAR RATING ───────────────────────────────────────
                    stars = hotel.find_elements(
                        By.CSS_SELECTOR,
                        "span.bdc459fcb4:not(.e2cec97860)"
                    )
                    hotel_info["star_rating"] = str(len(stars)) if stars else ""

                    # ── PROPERTY TYPE ─────────────────────────────────────
                    hotel_info["property_type"] = get_property_type(
                        hotel, hotel_info["hotel_name"])

                    # ── ROOM TYPE ─────────────────────────────────────────
                    hotel_info["room_type"] = extract_text_safe(
                        hotel, "[data-testid='recommended-units'] h4")

                    # ── FREE CANCELLATION ─────────────────────────────────
                    try:
                        card_text = hotel.text
                        hotel_info["free_cancellation"] = (
                            "Yes" if "Free cancellation" in card_text else "No"
                        )
                    except:
                        hotel_info["free_cancellation"] = "No"

                    # ── MEAL PLAN ─────────────────────────────────────────
                    try:
                        meal_elem = hotel.find_element(
                            By.CSS_SELECTOR, "span.d651523034")
                        hotel_info["meal_plan"] = meal_elem.text.strip()
                    except:
                        hotel_info["meal_plan"] = ""

                    # ── AMENITIES (scroll first for lazy loading) ─────────
                    hotel_info["amenities"] = extract_amenities(driver, hotel)

                    # ── HOTEL URL ─────────────────────────────────────────
                    try:
                        link = hotel.find_element(By.CSS_SELECTOR, "a")
                        hotel_info["hotel_url"] = link.get_attribute("href")
                    except:
                        pass

                    # ── SAVE ──────────────────────────────────────────────
                    if hotel_info["hotel_name"]:
                        hotels_data.append(hotel_info)
                        print(f"  {idx+1}. {hotel_info['hotel_name'][:50]}"
                              f" | {hotel_info['amenities'][:40] or 'no amenities'}")

                except Exception as e:
                    print(f"  Error on hotel {idx+1}: {e}")
                    continue

            # ── NEXT PAGE ─────────────────────────────────────────────────
            try:
                next_btn = driver.find_element(
                    By.CSS_SELECTOR, "[data-testid='pagination-next']")
                classes = next_btn.get_attribute("class") or ""
                outer   = next_btn.get_attribute("outerHTML") or ""
                if "disabled" in classes or "aria-disabled" in outer:
                    print("No more pages")
                    break
                next_btn.click()
                time.sleep(random.randint(3, 7))
                page_num += 1
            except:
                print("No next button found")
                break

    except Exception as e:
        print(f"Error: {e}")
    finally:
        driver.quit()

    return hotels_data


def save_to_csv(hotels, filename):
    if not hotels:
        print("No data to save")
        return

    fieldnames = [
        "city", "hotel_name", "price", "currency",
        "num_adults", "num_children", "num_nights",
        "rating", "reviews_count", "location",
        "distance_from_center", "star_rating", "property_type",
        "room_type", "free_cancellation", "meal_plan",
        "amenities", "hotel_url", "scraped_date"
    ]

    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(hotels)

    print(f"Saved {len(hotels)} hotels to {filename}")


# ── CITY CONFIGURATIONS ───────────────────────────────────────────────────────
CITIES = [
    {
        "name":     "Cairo",
        "ss":       "Cairo%2C+Egypt",
        "filename": "cairo_hotels.csv"
    },

     {
         "name":     "Alexandria",
         "ss":       "Alexandria%2C+Egypt",
         "filename": "alexandria_hotels.csv"
     },
     {
         "name":     "Hurghada",
         "ss":       "Hurghada%2C+Egypt",
         "filename": "hurghada_hotels.csv"
    },
     {
         "name":     "Sharm El Sheikh",
         "ss":       "Sharm+El+Sheikh%2C+Egypt",
         "filename": "sharm_hotels.csv"
    },
     {
         "name":     "Luxor",
         "ss":       "Luxor%2C+Egypt",
         "filename": "luxor_hotels.csv"
     },
     {
         "name":     "Aswan",
         "ss":       "Aswan%2C+Egypt",
         "filename": "aswan_hotels.csv"
     },
     {
         "name":     "Marsa Matruh",
         "ss":       "Marsa+Matruh%2C+Egypt",
         "filename": "marsamatruh_hotels.csv"
     },
     {
         "name":     "Dahab",
         "ss":       "Dahab%2C+Egypt",
         "filename": "dahab_hotels.csv"
     },
     {
            "name":     "El Gouna",
            "ss":       "El+Gouna%2C+Egypt",
            "filename": "elgouna_hotels.csv"
     },
     {
            "name":     "Marsa Alam",
            "ss":       "Marsa+Alam%2C+Egypt",
            "filename": "marsaalam_hotels.csv"
     },
     {
            "name":     "Siwa Oasis",
            "ss":       "Siwa+Oasis%2C+Egypt",
            "filename": "siwa_hotels.csv"
     },
     {
            "name":     "Fayoum",
            "ss":       "Fayoum%2C+Egypt",
            "filename": "fayoum_hotels.csv"
     },
     {
            "name":     "Ain Sokhna",
            "ss":       "Ain+Sokhna%2C+Egypt",
            "filename": "ainsokhna_hotels.csv"
     },
     {
            "name":     "El Alamein",
            "ss":       "El+Alamein%2C+Egypt",
            "filename": "elalamein_hotels.csv"
     },
     {
            "name":     "Sohag",
            "ss":       "Sohag%2C+Egypt",
            "filename": "sohag_hotels.csv"
     }
    

]


if __name__ == "__main__":
    print("=" * 50)
    print(f"Booking.com Hotel Scraper")
    print(f"Dates:    {CHECKIN} → {CHECKOUT} ({NUM_NIGHTS} night/s)")
    print(f"Guests:   {NUM_ADULTS} adults, {NUM_CHILDREN} children")
    print("=" * 50)

    all_hotels = []

    for city in CITIES:
        print(f"\nScraping: {city['name']}")
        url = build_url(city["ss"])

        hotels = scrape_hotels(
            city_name=city["name"],
            url=url,
            max_pages=10
        )

        if hotels:
            save_to_csv(hotels, city["filename"])
            print(f"Total {city['name']} hotels scraped: {len(hotels)}")
            all_hotels.extend(hotels)
        else:
            print(f"No hotels found for {city['name']}")

    # Save combined file when scraping multiple cities
    if len(CITIES) > 1 and all_hotels:
        save_to_csv(all_hotels, "all_egypt_hotels.csv")
        print(f"\nCombined file: {len(all_hotels)} total hotels")

    print("\nDone!")

Booking.com Hotel Scraper
Dates:    2026-06-01 → 2026-06-02 (1 night/s)
Guests:   2 adults, 0 children

Scraping: Cairo
Starting driver...
Loading https://www.booking.com/searchresults.html?ss=Cairo%2C+Egypt&lang=en-us&checkin=2026-06-01&checkout=2026-06-02&group_adults=2&no_rooms=1&group_children=0&dest_type=city...
Page loaded, searching for hotels...

--- Page 1 ---
Found 50 hotels on page 1
  1. Hilton Cairo Grand Nile | no amenities
  2. Agor-Cairo Hotel Downtown | no amenities
  3. The Treehouse | no amenities
  4. Royal Hotel Downtown | no amenities
  5. New Top Pyramids Hotel | no amenities
  6. gold cairo downtown | no amenities
  7. Hotel Comfort Giza | no amenities
  8. cairoooo houb | no amenities
  9. Lotus Hotel Downtown - Tahrir plaza | no amenities
  10. Pyramids New Locanda Hotel | no amenities
  11. Royal Crown Pyramids View | no amenities
  12. Ramses Hilton Hotel & Casino | no amenities
  13. GAR Hotel Cairo Downtown | no amenities
  14. New pyramids View & desert M

KeyboardInterrupt: 

: 